# Addressability Cost: what it takes to read one local cell of an infinite whole

**Dean Kulik / QuHarmonics — NEXUS.** Self-contained; run top to bottom.

## What this establishes

A structured infinite object can be read at a distant address *without materializing the
prefix*. That much is known for π in base 16 (BBP). This notebook separates that property
into three independent factors, measures each, and corrects two claims made along the way.

**The structure.** Reading local cell `n` of a whole `X` in base `b` is always three operations:

| | operation | what it is |
|---|---|---|
| **Fold** | each shifted term → `a_k·b^n mod q_k` | unbounded accumulation reduced to finite residue state |
| **Address** | `X → b^n·X` | brings depth `n` to the phase boundary |
| **Boundary** | `dist(frac(b^n X), d/b)` | whether the phase sits inside one cell |

**The three results.**

1. **Direct addressing does not require base alignment.** Tested on `e = Σ1/k!`, which has no
   radix scale at all. Hex digits extract correctly to n=2000. This *falsified* an earlier claim
   that the mechanism is "representation action = address action inverse" — that is the
   π-specific instance, not the mechanism.

2. **A mismatched radix costs, it does not block.** Decimal digits of π extract directly from
   the *base-16* BBP series, verified at six addresses to n=1000. The literature's
   "no base-10 BBP formula for π is known" is a statement about cheap structure, not accessibility.

3. **The cost is modulus growth, and nothing else.** Three measured growth laws. The boundary
   factor turns out to be O(1) on average, so it drops out.

**The surviving criterion:** *direct local readout exists when each term's contribution under the
address shift reduces to a residue whose modulus grows slowly enough to compute, and the tail
contracts.* Alignment buys exactly one thing — it removes the scale factor from the modulus,
taking it from exponential in k to logarithmic in k.

## Library — everything needed, nothing imported from elsewhere

In [1]:
import time, json
from math import log
from collections import Counter

try:
    from mpmath import mp
    import mpmath
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable,"-m","pip","install","-q","mpmath","--break-system-packages"])
    from mpmath import mp
    import mpmath

P = 400          # guard bits for integer-scaled accumulation

# ---------------------------------------------------------------- reference digits
def ref_hex(const, n_digits):
    """hex digits of frac(const), computed independently of any extraction method."""
    mp.dps = int(n_digits*1.21) + 60
    x = const()
    f = x - int(x)
    out = []
    for _ in range(n_digits):
        f *= 16; d = int(f); out.append('%x' % d); f -= d
    return ''.join(out)

# ---------------------------------------------- pi, BBP series, read in base 16 (ALIGNED)
BBP = ((1, 4), (4, -2), (5, -1), (6, -1))

def pi_hex_digit(n):
    """floor(16*frac(16^n pi)). Residues above the seam, geometric tail below."""
    total = 0; modbits = 0
    for k in range(0, n+1):                       # ABOVE: 16^(n-k) is an integer
        for j, c in BBP:
            m = 8*k + j; modbits += m.bit_length()
            total += c * ((pow(16, n-k, m) << P) // m)
    for k in range(n+1, n+40):                    # BELOW: contracting tail
        for j, c in BBP:
            total += c * ((1 << P) // (16**(k-n) * (8*k+j)))
    frac = total & ((1 << P) - 1)
    return (frac*16) >> P, n+1, modbits

# ------------------------------------------- e = sum 1/k!, read in base 16 (NO radix scale)
def e_hex_digit(n):
    """The non-circular test: e's series has no b^-k factor anywhere."""
    total = 0; modbits = 0; kf = 1; k = 0; tgt = 16**n
    while True:
        if k: kf *= k
        if kf > tgt: break
        modbits += kf.bit_length()
        total += (pow(16, n, kf) << P) // kf      # fold: 16^n mod k!
        k += 1
    K = k; kft = kf                               # kf == K! and term K was NOT yet added
    kk = K
    while True:                                   # factorial tail, superexponential
        t = (tgt << P) // kft
        total += t
        if t == 0: break
        kk += 1; kft *= kk
    frac = total & ((1 << P) - 1)
    return (frac*16) >> P, K, modbits

# --------------------------------- pi, BBP series, read in base 10 (MISMATCHED radix)
def pi_dec_digit(n):
    """floor(10*frac(10^n pi)) from the base-16 series. Modulus retains 16^k."""
    K = int(n*log(10)/log(16)) + 2
    total = 0; modbits = 0; tenn = 10**n
    for k in range(0, K+1):
        p16 = 16**k
        for j, c in BBP:
            m = p16*(8*k+j); modbits += m.bit_length()
            total += c * ((pow(10, n, m) << P) // m)
    for k in range(K+1, K+1+120):
        p16 = 16**k
        if (tenn << P)//p16 == 0: break
        for j, c in BBP:
            total += c * ((tenn << P) // (p16*(8*k+j)))
    frac = total & ((1 << P) - 1)
    return (frac*10) >> P, K, modbits

# ------------------------------------------------------------- leakage estimator
def _H(counts):
    n = sum(counts.values())
    return -sum(c/n*log(c/n, 2) for c in counts.values() if c)

def excess_entropy(seq, L):
    """empirical I(past_L ; future_L), bits. Biased UP; always read against an iid control."""
    U, V, J = Counter(), Counter(), Counter()
    for i in range(len(seq)-2*L+1):
        u = tuple(seq[i:i+L]); v = tuple(seq[i+L:i+2*L])
        U[u] += 1; V[v] += 1; J[(u, v)] += 1
    return _H(U) + _H(V) - _H(J)

# ------------------------------------------------------------- boundary margin
def guard_runs(digits, pad=40):
    """run of 0s or fs starting after each address = extra hex digits needed to certify it."""
    N = len(digits) - pad
    runs = Counter()
    for n in range(N):
        r = 0; c0 = digits[n+1]
        if c0 in '0f':
            while digits[n+1+r] == c0: r += 1
        runs[r] += 1
    return runs, N

print("library ready")

library ready


## 1. Reference expansions (independent of every extraction method below)

In [2]:
PI_HEX = ref_hex(lambda: mpmath.pi, 20000)
E_HEX  = ref_hex(lambda: mpmath.e,  2100)
print("pi =  3." + PI_HEX[:24] + "...")
print("e  =  2." + E_HEX[:24]  + "...")

pi =  3.243f6a8885a308d313198a2e...
e  =  2.b7e151628aed2a6abf715880...


## 2. Aligned case — π in base 16

`16^n · 16^-k = 16^(n-k)`, an integer for `k ≤ n`. The moduli are `8k+j`: **logarithmic** in k.

In [3]:
print(" n        got  ref")
for n in (0, 1, 2, 9, 99, 999, 4999):
    d, K, mb = pi_hex_digit(n)
    print(f" {n:<8} {'%x'%d:^4} {PI_HEX[n]:^4}  {'ok' if '%x'%d==PI_HEX[n] else 'MISMATCH'}")

 n        got  ref
 0         2    2    ok
 1         4    4    ok
 2         3    3    ok
 9         5    5    ok
 99        c    c    ok
 999       3    3    ok
 4999      1    1    ok


## 3. The non-circular test — e in base 16

The criterion above was built entirely on π/BBP. `e = Σ1/k!` has **no radix scale**: there is no
`16^-k` to invert. If the criterion were tailored to π it has no business working here.

The fold is `16^n mod k!` and the tail contracts superexponentially.

In [4]:
print(" n        got  ref     terms above seam")
for n in (0, 1, 5, 50, 500, 1500, 2000):
    d, K, mb = e_hex_digit(n)
    print(f" {n:<8} {'%x'%d:^4} {E_HEX[n]:^4}  {'ok' if '%x'%d==E_HEX[n] else 'MISMATCH':8} {K:>8}")

 n        got  ref     terms above seam
 0         b    b    ok              2
 1         7    7    ok              4
 5         1    1    ok             10
 50        8    8    ok             48
 500       d    d    ok            295
 1500      a    a    ok            741
 2000      5    5    ok            947


**Result: it works.** So "representation action = address action inverse" is *not* the mechanism —
it was the π-specific instance. What is actually required is only that each shifted term reduce to
a residue modulo its own integer denominator.

## 4. The mismatched case — π in base 10 from the base-16 series

Now the modulus retains the scale factor: `10^n mod (16^k·(8k+j))`, whose bit length is ~4k —
**linear** in k instead of logarithmic.

Note the indexing: `frac(10^n π)` brings decimal digit `d[n]` to the lead, **not** `d[n-1]`.
Getting this wrong produces a false negative at every single address.

In [5]:
mp.dps = 1400
PI_DEC = mpmath.nstr(mpmath.pi, 1300, strip_zeros=False).replace('3.', '', 1)
print(" n        got  d[n-1]  d[n]")
for n in (1, 5, 50, 200, 500, 1000):
    g, K, mb = pi_dec_digit(n)
    tag = '-> d[n]' if str(g)==PI_DEC[n] else ('-> d[n-1]' if str(g)==PI_DEC[n-1] else 'NEITHER')
    print(f" {n:<8} {g:^4} {PI_DEC[n-1]:^7} {PI_DEC[n]:^5}  {tag}")

 n        got  d[n-1]  d[n]
 1         4      1      4    -> d[n]
 5         2      9      2    -> d[n]
 50        5      0      5    -> d[n]
 200       4      6      4    -> d[n]
 500       9      2      9    -> d[n]
 1000      3      9      3    -> d[n]


**Result: decimal digits of π are directly extractable from the base-16 series.** The prefix is
never built. A mismatched radix is a cost, not a wall — which is consistent with the literature:
"no base-10 BBP formula for π is known" constrains *cheap* structure, not accessibility.

## 5. Cost profile — the three growth laws

In [6]:
print(" method                 n      K       modulus bits      time")
rows = {}
for name, fn in (("pi BBP base 16", pi_hex_digit),
                 ("e factorial base 16", e_hex_digit),
                 ("pi BBP under base 10", pi_dec_digit)):
    for n in (500, 1000):
        t0 = time.time(); _, K, mb = fn(n); dt = time.time()-t0
        rows[(name, n)] = mb
        print(f" {name:22s}{n:<7}{K:<8}{mb:>13}    {dt:.3f}s")

print("\n doubling ratio n=500 -> n=1000  (2.0 ~ linear, 4.0 ~ quadratic)")
for name in ("pi BBP base 16", "e factorial base 16", "pi BBP under base 10"):
    print(f"   {name:24s}{rows[(name,1000)]/rows[(name,500)]:.2f}")
print(f"\n alignment advantage at n=1000: "
      f"{rows[('pi BBP under base 10',1000)]/rows[('pi BBP base 16',1000)]:.0f}x fewer modulus bits")

 method                 n      K       modulus bits      time
 pi BBP base 16        500    501             22002    0.002s
 pi BBP base 16        1000   1001            47958    0.003s
 e factorial base 16   500    295            263388    0.002s
 e factorial base 16   1000   526            952035    0.005s
 pi BBP under base 10  500    417           1412466    0.006s
 pi BBP under base 10  1000   832           5583670    0.026s

 doubling ratio n=500 -> n=1000  (2.0 ~ linear, 4.0 ~ quadratic)
   pi BBP base 16          2.18
   e factorial base 16     3.61
   pi BBP under base 10    3.95

 alignment advantage at n=1000: 116x fewer modulus bits


| representation | modulus `q_k` | bits each | total ledger |
|---|---|---|---|
| π, BBP base 16 | `8k+j` | `O(log k)` | **`O(n log n)`** |
| e, factorial | `k!` | `Θ(k log k)` | **`~O(n²)`** |
| π, base-10 shift | `16^k(8k+j)` | `Θ(k)` | **`O(n²)`, ~116× constant** |

All three are **representation-direct** (no prefix). Only the aligned case is **resource-direct**.

## 6. The boundary — and why it drops out

`δ_n = dist(frac(16^n π), nearest d/16)` is small exactly when the digits *after* n run as 0s
(just above a boundary) or fs (just below). That run length **is** the extra precision needed.
Equidistribution predicts `P(run ≥ r) = 2·16^-r`.

In [7]:
runs, N = guard_runs(PI_HEX)
print(f" guard hex digits needed   (N = {N} addresses)")
print(f" {'r':>3} {'count':>9} {'measured P':>12} {'2*16^-r':>10}")
for r in sorted(runs):
    if r > 5: break
    ge = sum(v for k, v in runs.items() if k >= r)
    print(f" {r:>3} {runs[r]:>9} {ge/N:>12.5f} {(2*16.0**-r if r else 1.0):>10.5f}")
mean_guard = sum(k*v for k, v in runs.items())/N
print(f"\n mean guard digits: {mean_guard:.3f}   worst in range: {max(runs)}")
print(" -> O(1) on average, O(log_16 n) worst case over n addresses: NOT the bottleneck.")

 guard hex digits needed   (N = 19960 addresses)
   r     count   measured P    2*16^-r
   0     17423      1.00000    1.00000
   1      2377      0.12710    0.12500
   2       150      0.00802    0.00781
   3        10      0.00050    0.00049

 mean guard digits: 0.136   worst in range: 3
 -> O(1) on average, O(log_16 n) worst case over n addresses: NOT the bottleneck.


## 7. Leakage — the fold is invisible, and that is not what makes addressing work

Separate measurement, kept because it rules out a tempting wrong conclusion. π's hex stream and
the sum-bit stream of a balanced binary adder (whose hidden carry is a lattice fold of the entire
low-order past) both leak nothing. But the period-7 control leaks maximally **and** is trivially
addressable — so leakage is not the axis.

In [8]:
import random
rng = random.Random(7)
M = 60000
pi_sym  = [int(c, 16) for c in PI_HEX[:M]] if len(PI_HEX) >= M else [int(c,16) for c in PI_HEX]
M = len(pi_sym)
iid_sym = [rng.randrange(16) for _ in range(M)]
c = 0; bits = []
for _ in range(M*4):
    a, b = rng.randrange(2), rng.randrange(2)
    bits.append(a ^ b ^ c); c = 1 if a+b+c >= 2 else 0
carry_sym = [sum(bits[4*i+j] << j for j in range(4)) for i in range(M)]
per_sym = [i % 7 for i in range(M)]

print(f" N = {M}, alphabet 16")
print(f" {'stream':28s}{'E(L=1)':>9}")
for name, s in (("pi hex digits", pi_sym), ("iid hex (noise floor)", iid_sym),
                ("carry sum-bits", carry_sym), ("CONTROL period-7", per_sym)):
    print(f" {name:28s}{excess_entropy(s,1):>9.4f}")

 N = 20000, alphabet 16
 stream                         E(L=1)
 pi hex digits                  0.0087
 iid hex (noise floor)          0.0072
 carry sum-bits                 0.0076


 CONTROL period-7               2.8074


## What stands, and what does not

**Stands (measured here):**
- Direct addressing on π base 16, e base 16, π base 10 — all verified against independent expansions.
- Three distinct ledger growth laws, confirmed by doubling ratio.
- Boundary guard cost is O(1) average, geometric tail matching 2·b^-r over four orders of magnitude.
- Alignment's single benefit: it removes the scale factor from the modulus.

**Corrected during this work:**
- "Representation action = address action inverse" — refuted by e.
- "Mismatched radix collapses direct addressing" — refuted by the base-10 π extraction.
- A false MISMATCH at every decimal address, caused by off-by-one reference indexing, not by the method.

**Not established:**
- Normality of π. The leakage column is 60k digits consistent with zero, nothing stronger.
- Whether `O(n log n)` is optimal for any constant, or which constants admit log-modulus folds.
- Any claim that this structure governs physical systems. It is measured for three number
  representations and nothing else.